In [1]:
import pandas as pd

df_rpk = pd.read_csv('All_source_plates_mapped_rpk_column_names_standardized_unique_identifiers_for_duplicates_with_hbdb_plate_samples_dropped_less_100k_readcounts_071426.csv').set_index('peptide')

In [2]:
df_rpk.head(2)

,C-452-2_5_H3_1,C-109-3_4_E7_1,C-109-3_4_E7_2,C-111-3_3_I6_1,C-111-3_3_I6_2,C-112-1_3_B3_1,C-112-1_3_B3_2,C-112-2_7_G7_1,C-112-2_7_G7_2,C-114-2_7_B1_1,...,HBDB-109_1,HBDB-109_2,HBDB-111_1,HBDB-111_2,HBDB-115_1,HBDB-115_2,mAb_1_7,mAb_1_8,mAb_2_7,mAb_2_8
peptide,,,,,,,,,,,,,,,,,,,,,
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_1,0.0,0.0,0.0,0.870155,0.0,0.0,0.918952,0.368368,0.000000,0.0,...,0.000000,0.0,0.0,0.255092,0.00000,0.0,0.0,0.0,0.178464,0.59977
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_2,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.580242,0.0,...,0.214736,0.0,0.0,0.255092,0.68292,0.0,0.0,0.0,0.000000,0.00000


### Take average of technical replicates

In [3]:
# For C-/H-/S- samples: strip one trailing _N (technical replicate suffix)
# For AG/Canary/mAb: strip ALL trailing _N or _sN segments so they all collapse to their base name
def get_prefix(col):
    if re.match(r'^(AG|Canary|mAb)', col):
        return re.sub(r'(_s?\d+)+$', '', col)
    else:
        return re.sub(r'_\d+$', '', col) #this starts removing from the end of the string, $ means end, \d one or more digits, _ underscore

import re
prefix = df_rpk.columns.map(get_prefix)

## first do a correlation analysis of a subset of random technical replicates (same prefix)

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Find all prefixes that have >= 2 replicates
prefix_series = pd.Series(prefix.values, index=df_rpk.columns)
replicated = prefix_series[prefix_series.duplicated(keep=False)]
replicated_groups = replicated.groupby(replicated).apply(lambda g: g.index.tolist())

# Pick 5 random replicated groups
random.seed(42)
sampled_groups = random.sample(list(replicated_groups.index), min(10, len(replicated_groups)))

print(f'Total groups with >= 2 replicates: {len(replicated_groups)}')
print('Sampled groups:', sampled_groups)

Total groups with >= 2 replicates: 314
Sampled groups: ['C-150-3_7_C1', 'C-114-3_4_F3', 'C-510-3_6_D4', 'C-489-3_6_C3', 'C-482-3_4_I8', 'C-438-1_4_B2', 'C-147-2_1_F5', 'S-168_SA2_B3', 'C-139-2_6_D5', 'S-469_SA2_I8']


In [10]:
replicated_groups

AG              [AG_2_2, AG_2_1, AG_2_3, AG_s4_1, AG_s4_2, AG_...
C-109-1_5_B3                     [C-109-1_5_B3_1, C-109-1_5_B3_2]
C-109-3_4_E7                     [C-109-3_4_E7_1, C-109-3_4_E7_2]
C-110-2_6_A2                     [C-110-2_6_A2_1, C-110-2_6_A2_2]
C-111-3_3_I6                     [C-111-3_3_I6_1, C-111-3_3_I6_2]
                                      ...                        
S-502_SA4_I1                     [S-502_SA4_I1_1, S-502_SA4_I1_2]
S-507_SA4_E1                     [S-507_SA4_E1_1, S-507_SA4_E1_2]
S-509_SA3_H6                     [S-509_SA3_H6_1, S-509_SA3_H6_2]
S-514_SA3_C8                     [S-514_SA3_C8_1, S-514_SA3_C8_2]
mAb             [mAb_1, mAb_2, mAb_3, mAb_4, mAb_1_1_1, mAb_1_...
Length: 314, dtype: object

## take the average value for the technical replicates

In [6]:
df_rpk_avg = df_rpk.T.groupby(prefix, sort=False).mean().T

In [7]:
df_rpk_avg.shape

(90132, 963)

In [9]:
df_rpk_avg.to_csv('rpk_df_technical_replicates_samples_passing_readcounts_100k_all_averaged_071626.csv')

### generate df for hbdb avg values

In [11]:
import re

df_hbdb = df_rpk_avg.filter(regex=('^HBDB-'))

In [13]:
df_hbdb.to_csv('hbdb_pass_100k_techreps_averaged_071626.csv')